# 02 — Astra weight construction and paired Wall comparison
Run notebook 01 first. Use the same repository commit and Drive experiment ID.

This notebook checks the baseline handoff, freezes a runtime-specific inspection
packet, lets your selected Astra API model inspect weights/source, and accepts
one controller submission. It then evaluates those frozen weights on the exact
same Wall episodes. A fixed-budget recheck in this session gives a fairer latency
comparison if Colab hardware changed.

The model receives checkpoint/source inspection tools only. Baseline scores,
episode manifests, world-model forward passes, rollouts, gradients and evaluation
feedback are not tools. API generation is billable; an attempt is not retried or
repaired automatically. Reusing an already accepted submission makes no API call.


In [ ]:
# Edit these in BOTH notebooks. Use the same Git commit and Drive experiment ID.
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
PROJECT_REF = "main"  # Prefer the exact commit containing these notebooks.
EXPERIMENT_ID = "wall-pilot-001"
DRIVE_FOLDER = "/content/drive/MyDrive/wsp-stopper"


In [ ]:
from pathlib import Path
import sys, subprocess, os, json, hashlib, shutil
from google.colab import drive
assert 'YOUR_' not in REPO_URL, 'Set REPO_URL to your repository URL'
assert '/' not in EXPERIMENT_ID and '..' not in EXPERIMENT_ID
# For a private repo, clone it manually using your preferred credentials first;
# never embed a GitHub token in REPO_URL or save one in notebook output.
drive.mount('/content/drive')
PROJECT = Path('/content/wsp-stopper')
if not (PROJECT/'.git').exists():
    subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
else:
    assert subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()==REPO_URL
subprocess.run(['git','-C',str(PROJECT),'checkout',PROJECT_REF],check=True)
os.chdir(PROJECT)
PROJECT_COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
required=['core.py','cli.py','agent.py','packet.py','inspection.py','wall_setup.py','cem_hook.py','upstream_patch.py']
assert all((PROJECT/'src/wsp_stopper'/f).exists() for f in required), 'Commit the repaired complete starter first'
# Retain Colab's preinstalled matched torch/torchvision pair; do not install the
# upstream all-simulator pyproject (which declares Python 3.10 only).
subprocess.run([sys.executable,'-m','pip','install','--quiet',
    'einops==0.8.1','tensordict==0.9.1','gym==0.23.1','omegaconf==2.3.0',
    'PyYAML>=6,<7','scipy','matplotlib','tqdm','pandas'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.', '--no-deps','--quiet'],check=True)
sys.path.insert(0,str(PROJECT/'src'))
import torch, torchvision, tensordict
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2,7)
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then rerun'
print('Python:',sys.version.split()[0], 'torch:',torch.__version__,'vision:',torchvision.__version__)
print('GPU:',torch.cuda.get_device_name(0),'project:',PROJECT_COMMIT)
EXPERIMENT = Path(DRIVE_FOLDER)/EXPERIMENT_ID
EXPERIMENT.mkdir(parents=True,exist_ok=True)
WORK = PROJECT/'assets/wall-colab-v1'
BASE_PACKET = PROJECT/'packets/wall-colab-base-v1'


In [ ]:
# This exact helper is embedded in both notebooks: no extra repo files required.
RUNTIME = Path('/content/wsp_wall_notebook_runtime.py')
RUNTIME_SOURCE = '"""Shared Colab runtime. Embedded byte-identically in both notebooks.\nRuns pinned upstream model/planner classes with a small dependency surface.\nNo evaluation files are included in the generator\'s allowlisted packet.\n"""\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport types\nfrom typing import Callable, List, Union\nfrom abc import ABC, abstractmethod\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom einops import rearrange, repeat\nfrom tensordict import TensorDict\nimport yaml\nfrom wsp_stopper.core import digest, write_json\n\n\ndef definitions(path, names, namespace=None):\n    """Execute only named classes from pinned trusted source, no unrelated imports.\n    This keeps their method bodies verbatim and avoids simulator/training imports.\n    """\n    tree = ast.parse(Path(path).read_text())\n    selected = [n for n in tree.body if isinstance(n, ast.ClassDef) and n.name in names]\n    if {n.name for n in selected} != set(names):\n        raise ValueError(f\'Missing classes in {path}\')\n    env = dict(globals())\n    if namespace: env.update(namespace)\n    env[\'__name__\'] = \'wsp_notebook_upstream\'\n    exec(compile(ast.Module(body=selected,type_ignores=[]),str(path),\'exec\'),env)\n    return [env[name] for name in names]\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef sync():\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n\n\ndef configure(work):\n    work=Path(work).resolve()\n    up=work/\'jepa-wms\'\n    sys.path.insert(0,str(up))\n    os.environ[\'WSP_DINOV2_SOURCE\']=str(work/\'dinov2\')\n    os.environ[\'WSP_DINOV2_WEIGHTS\']=str(work/\'encoder.pth\')\n    torch.set_num_threads(2)\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False\n    return work,up\n\n\ndef make_environment(up):\n    # Direct DotWall construction avoids unrelated mujoco/robocasa/dataset imports.\n    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall\n    source=ast.parse((up/\'evals/simu_env_planning/envs/wall_gym_wrap.py\').read_text())\n    expr=next(n.value for n in source.body if isinstance(n,ast.Assign)\n        and any(isinstance(t,ast.Name) and t.id==\'DEFAULT_CFG\' for t in n.targets))\n    from evals.simu_env_planning.envs.wall_env.data.wall import WallDatasetConfig\n    cfg=eval(compile(ast.Expression(expr),\'wall_default_cfg\',\'eval\'),{\'WallDatasetConfig\':WallDatasetConfig})\n    cfg.device=\'cpu\'\n    # Match wrapper default layout generation; generate_random_state alternates travel direction.\n    return DotWall(rng=np.random.default_rng(0),wall_config=cfg,device=\'cpu\')\n\n\ndef create_tasks(work, output, count=10, seed_start=1000):\n    _,up=configure(work)\n    if Path(output).exists(): raise ValueError(\'Task manifest exists; reuse it instead of overwriting\')\n    seed_all(123)\n    env=make_environment(up)\n    episodes=[]\n    for i in range(count):\n        s,g=env.generate_random_state(seed_start+i)\n        episodes.append(dict(id=i,environment_seed=seed_start+i,planner_seed=900000+i,\n            start=s.tolist(),goal=g.tolist(),wall_x=float(env.wall_x),door_y=float(env.hole_y)))\n    write_json(output,dict(schema=1,task=\'Wall fixed layout cross-room navigation\',episodes=episodes))\n\n\nclass Preprocessor:\n    def __init__(self, stats, device):\n        from torchvision.transforms import Normalize\n        self.mean=torch.tensor(stats[\'proprio_mean\'])\n        self.std=torch.tensor(stats[\'proprio_std\'])\n        self.transform=Normalize([.5]*3,[.5]*3)\n    def normalize_proprios(self,x): return (x-self.mean)/self.std\n\n\ndef load_model(work,device):\n    _,up=configure(work)\n    # These modules have only small torch/einops dependencies.\n    from app.plan_common.models.vit import ViTPredictor\n    from app.plan_common.models.prop_embedding import ProprioceptiveEmbedding\n    from app.plan_common.models.dino import DinoEncoder\n    from app.vjepa_wm.video_wm import VideoWM\n    cfg=yaml.safe_load((work/\'wall.eval.yaml\').read_text())\n    kwargs=cfg[\'model_kwargs\'][\'pretrain_kwargs\']\n    stats_ast=ast.parse((up/\'app/plan_common/datasets/__init__.py\').read_text())\n    stats=next(ast.literal_eval(n.value)[\'wall\'] for n in stats_ast.body if isinstance(n,ast.Assign)\n        and any(isinstance(t,ast.Name) and t.id==\'DATA_STATS\' for t in n.targets))\n    encoder=DinoEncoder(\'dinov2_vits14\',\'x_norm_patchtokens\').to(device)\n    predictor=ViTPredictor(depth=6,heads=16,mlp_dim=2048,dropout=.1,\n        num_patches=256,num_frames=4,dim=404,use_sdpa=True).to(device)\n    def embedding(dim):\n        return ProprioceptiveEmbedding(num_frames=4,tubelet_size=1,tokens_per_step=1,\n            in_chans=dim,embed_dim=10,shift_input=False,mlp_dims=[10],use_mlp=False).to(device)\n    action,prop=embedding(10),embedding(2)\n    ckpt=torch.load(work/\'wall.pth.tar\',map_location=\'cpu\',weights_only=True)\n    report={}\n    for name,model in [(\'predictor\',predictor),(\'action_encoder\',action),(\'proprio_encoder\',prop)]:\n        state={k.replace(\'module.\',\'\'):v for k,v in ckpt[name].items()}\n        state={k.replace(\'state_encoder.weight\',\'proprio_encoder.weight\').replace(\'state_encoder.bias\',\'proprio_encoder.bias\'):v for k,v in state.items()}\n        # Some upstream versions omit regenerated causal masks from state_dict.\n        msg=model.load_state_dict(state,strict=False)\n        allowed=[k for k in msg.missing_keys if k.endswith(\'.bias\') and \'transformer.layers\' in k]\n        if msg.unexpected_keys or set(msg.missing_keys)-set(allowed):\n            raise ValueError(f\'Checkpoint mismatch: {name}: {msg}\')\n        report[name]=dict(missing_regenerated_masks=allowed,unexpected=msg.unexpected_keys)\n    video=VideoWM(encoder=encoder,predictor=predictor,action_encoder=action,proprio_encoder=prop,\n        enc_type=\'dino\',pred_type=\'dino_wm\',grid_size=16,tubelet_size_enc=1,img_size=224,\n        action_dim=10,proprio_dim=2,action_tokens=0,proprio_tokens=0,use_action=True,use_proprio=True,\n        action_skip=1,frameskip=5,action_conditioning=\'feature\',proprio_encoding=\'feature\',\n        action_encoder_inpred=False,proprio_encoder_inpred=False,batchify_video=True,\n        dup_image=False,normalize_reps=False,device=device,heads={},cfgs_loss={\'proprio_loss\':True})\n    Wrapper,=definitions(up/\'app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py\',[\'EncPredWM\'])\n    wrapped=Wrapper(video,10,Preprocessor(stats,device),ctxt_window=2).to(device).eval()\n    report[\'encoder\']=\'strict=True, local pinned source and tensors\'\n    return wrapped,stats,report\n\n\ndef observation(env,state,device):\n    from torchvision.transforms import Resize\n    obs,_=env.reset(location=torch.tensor(state,dtype=torch.float32))\n    image=Resize((224,224))(obs[\'visual\'])\n    return {\'visual\':image.unsqueeze(0).unsqueeze(0).to(device),\n            \'proprio\':obs[\'proprio\'].view(1,1,2).to(device)}\n\n\ndef make_planner(work,model,z_goal,iterations,population,elites,chunk,planner_seed,device):\n    up=work/\'jepa-wms\'\n    objective_file=up/\'evals/simu_env_planning/planning/planning/objectives.py\'\n    _,Objective=definitions(objective_file,[\'BaseMPCObjective\',\'ReprTargetDistMPCObjective\'])\n    planner_file=up/\'evals/simu_env_planning/planning/planning/planner.py\'\n    _,_,CEM=definitions(planner_file,[\'PlanningResult\',\'Planner\',\'CEMPlanner\'],\n        {\'NamedTuple\':__import__(\'typing\').NamedTuple,\'objectives\':SimpleNamespace(BaseMPCObjective=object)})\n    counts={\'candidate_transition_evals\':0,\'mean_transition_evals\':0,\'candidate_sequences\':0}\n    def counted_unroll(z,act_suffix=None):\n        counts[\'mean_transition_evals\']+=act_suffix.shape[0]*act_suffix.shape[1]\n        return model.unroll(z,act_suffix=act_suffix)\n    generator=torch.Generator(device=device).manual_seed(planner_seed)\n    p=CEM(counted_unroll,iterations=iterations,num_samples=population,num_elites=elites,\n        horizon=6,action_dim=10,var_scale=1,num_act_stepped=6,\n        local_generator=generator,decode_each_iteration=False,distribute_planner=False)\n    p.device=torch.device(device)\n    p.objective=Objective(None,z_goal,sum_all_diffs=False,alpha=.1)\n    # Identical microbatching for baseline and WSP. No RNG is consumed by model eval.\n    def costs(self,actions,z):\n        vals=[]\n        for j in range(0,actions.shape[1],chunk):\n            a=actions[:,j:j+chunk]\n            predicted=model.unroll(z,act_suffix=a)\n            vals.append(self.objective(predicted,a))\n            counts[\'candidate_transition_evals\']+=a.shape[0]*a.shape[1]\n            counts[\'candidate_sequences\']+=a.shape[1]\n        return torch.cat(vals)\n    p.cost_function=types.MethodType(costs,p)\n    return p,counts\n\n\ndef evaluate(work,tasks_path,out,iterations=30,population=300,elites=10,chunk=16,weights=None,allow_cpu=False):\n    if Path(out).exists(): raise ValueError(\'Result exists; use a new filename\')\n    if weights and iterations!=30: raise ValueError(\'Controller requires cap 30\')\n    if not (2<=elites<=population and chunk>=1 and 1<=iterations<=30): raise ValueError(\'Invalid budget\')\n    if not torch.cuda.is_available() and not allow_cpu: raise RuntimeError(\'Select a Colab GPU runtime\')\n    device=\'cuda:0\' if torch.cuda.is_available() else \'cpu\'\n    work,up=configure(work)\n    seed_all(123)\n    model,stats,load_report=load_model(work,device)\n    episodes=json.loads(Path(tasks_path).read_text())[\'episodes\']\n    amean=torch.tensor(stats[\'action_mean\']);astd=torch.tensor(stats[\'action_std\'])\n    if weights: os.environ[\'WSP_STOPPER_WEIGHTS\']=str(Path(weights).resolve())\n    else: os.environ.pop(\'WSP_STOPPER_WEIGHTS\',None)\n    # Warm up encoding/one rollout, without using any scored evaluation outcome.\n    seed_all(42);env=make_environment(up)\n    warm=model.encode(observation(env,[15.,20.],device))\n    with torch.inference_mode(): model.unroll(warm,torch.zeros(1,1,10,device=device))\n    sync()\n    rows=[]\n    for ep in episodes:\n        seed_all(ep[\'environment_seed\']);env=make_environment(up)\n        if float(env.wall_x)!=ep[\'wall_x\'] or float(env.hole_y)!=ep[\'door_y\']: raise ValueError(\'Wall layout mismatch\')\n        goal_obs=observation(env,ep[\'goal\'],device)\n        start_obs=observation(env,ep[\'start\'],device)\n        sync();begin=time.perf_counter()\n        with torch.inference_mode():\n            z=model.encode(start_obs);zg=model.encode(goal_obs)\n            planner,counters=make_planner(work,model,zg,iterations,population,elites,chunk,ep[\'planner_seed\'],device)\n            result=planner.plan(z)\n        sync();elapsed=time.perf_counter()-begin\n        raw=result.actions.detach().cpu().reshape(-1,2)*astd+amean\n        seed_all(ep[\'environment_seed\'])\n        env.reset(location=torch.tensor(ep[\'start\'],dtype=torch.float32))\n        positions=[ep[\'start\']]\n        any_success=False\n        for a in raw:\n            _,_,_,info=env.step(a)\n            pos=info[\'state\'].detach().cpu().numpy()\n            positions.append(pos.tolist())\n            any_success |= bool(np.linalg.norm(pos-np.asarray(ep[\'goal\']))<4.5)\n        distance=float(np.linalg.norm(np.asarray(positions[-1])-np.asarray(ep[\'goal\'])))\n        depth=int(planner.wsp_last_depth)\n        rows.append(dict(id=ep[\'id\'],success=distance<4.5,any_step_success=any_success,\n            final_distance=distance,depth=depth,planning_seconds=elapsed,\n            utility=float(distance<4.5)-.05*depth/30,\n            actions=raw.tolist(),positions=positions,stopper_trace=planner.wsp_last_trace,\n            **counters))\n        print(f"episode {ep[\'id\']}: success={distance<4.5}, depth={depth}, seconds={elapsed:.2f}",flush=True)\n        del result,planner,z,zg\n    config=dict(horizon=6,chunk=6,frameskip=5,raw_episode_steps=30,population=population,\n        elites=elites,microbatch=chunk,cap=30,proprio_weight=.1,success=\'final_position_distance < 4.5\')\n    result=dict(schema=1,tasks_sha256=digest(tasks_path),runtime_sha256=digest(__file__),\n        release_lock=json.loads(Path(\'config/wall.release.json\').read_text()),config=config,\n        fixed_iterations=iterations,weights_sha256=digest(weights) if weights else None,\n        model_load_report=load_report,torch_version=torch.__version__,\n        device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\',episodes=rows)\n    write_json(out,result)\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'cmd\',required=True)\n    t=sub.add_parser(\'tasks\');t.add_argument(\'--work\',required=True);t.add_argument(\'--out\',required=True)\n    t.add_argument(\'--count\',type=int,default=10);t.add_argument(\'--seed-start\',type=int,default=1000)\n    e=sub.add_parser(\'evaluate\');e.add_argument(\'--work\',required=True);e.add_argument(\'--tasks\',required=True);e.add_argument(\'--out\',required=True)\n    e.add_argument(\'--iterations\',type=int,default=30);e.add_argument(\'--population\',type=int,default=300)\n    e.add_argument(\'--elites\',type=int,default=10);e.add_argument(\'--microbatch\',type=int,default=16)\n    e.add_argument(\'--weights\');e.add_argument(\'--allow-cpu\',action=\'store_true\')\n    a=p.parse_args()\n    if a.cmd==\'tasks\': create_tasks(a.work,a.out,a.count,a.seed_start)\n    else: evaluate(a.work,a.tasks,a.out,a.iterations,a.population,a.elites,a.microbatch,a.weights,a.allow_cpu)\n\nif __name__==\'__main__\':main()\n'
RUNTIME.write_text(RUNTIME_SOURCE)
RUNTIME_SHA=hashlib.sha256(RUNTIME.read_bytes()).hexdigest()

def run_python(*args):
    subprocess.run([sys.executable,*map(str,args)],check=True,cwd=PROJECT)

from wsp_stopper.core import digest, write_json

def project_fingerprint():
    files=list((PROJECT/'src/wsp_stopper').glob('*.py'))+[PROJECT/'config/wall.release.json',PROJECT/'prompts/generate.md']
    return {str(p.relative_to(PROJECT)):digest(p) for p in sorted(files)}


In [ ]:
TASKS=EXPERIMENT/'tasks.json'
PROTOCOL=EXPERIMENT/'protocol.json'
BASELINE=EXPERIMENT/'baseline_k30.json'
assert all(p.exists() for p in [TASKS,PROTOCOL,BASELINE]), 'Finish notebook 01 first'
protocol=json.loads(PROTOCOL.read_text())
baseline_saved=json.loads(BASELINE.read_text())
assert protocol['project_commit']==PROJECT_COMMIT, 'Set PROJECT_REF to the commit recorded in protocol.json'
assert protocol['project_files']==project_fingerprint(), 'Project source changed'
assert protocol['runtime_sha256']==RUNTIME_SHA, 'The notebooks contain different runner versions'
assert protocol['tasks_sha256']==digest(TASKS)==baseline_saved['tasks_sha256']
assert baseline_saved['runtime_sha256']==RUNTIME_SHA
POPULATION=protocol['population'];ELITES=protocol['elites'];MICROBATCH=protocol['microbatch']
print('Paired handoff verified. Outcomes will not be included in the generator packet.')


In [ ]:
# Network + CPU tensor preparation only. No API call or world-model planning.
run_python('-m','wsp_stopper.wall_setup','prepare','--work',WORK)
if not BASE_PACKET.exists():
    run_python('-m','wsp_stopper.wall_setup','freeze','--work',WORK,'--out',BASE_PACKET)
run_python('-m','wsp_stopper.cli','verify',BASE_PACKET)
os.environ['WSP_TEST_UPSTREAM']=str(WORK/'jepa-wms')
run_python('-m','unittest','discover','-s','tests','-v')


## Freeze the exact runtime specification for generation
The new packet includes the notebook runner so Astra can inspect the actual
class loading, microbatching, preprocessing, final-success definition, and
execution code. It does not include task values or empirical baseline results.
This updates the earlier source-only preparation status after actual model
loading and baseline execution in notebook 01.


In [ ]:
from wsp_stopper.packet import freeze, verify
base_manifest=verify(BASE_PACKET)
GEN_PACKET=PROJECT/'packets/wall-colab-generation-v1'
spec=dict(base_manifest['specification'])
spec['planner']=dict(spec['planner'])
spec['planner'].update(num_samples=POPULATION,num_elites=ELITES,iterations=30)
spec['planner']['candidate_microbatch']=MICROBATCH
spec['success_predicate']='Final raw position L2 distance to goal < 4.5 after 30 simulator steps; no early success termination'
spec['validation_status']='Baseline executed and model tensor loading checked; no measured performance supplied'
spec['runtime_adapter']='Pinned upstream classes; class-only AST loading for EncPredWM/planner/objective to avoid unrelated imports; see notebook_runtime.py source'
spec['runtime_sha256']=RUNTIME_SHA
spec_path=PROJECT/'assets/wall-colab-v1/notebook.spec.json'
write_json(spec_path,spec)
sources=[BASE_PACKET/rel for rel in base_manifest['sources'] if not rel.endswith('wall.spec.json')]
sources += [RUNTIME,spec_path]
if not GEN_PACKET.exists():
    freeze(WORK/'inspection-state.pt','',sources,spec_path,
        dict(base_manifest['provenance'],notebook_runtime_sha256=RUNTIME_SHA),GEN_PACKET)
gen_manifest=verify(GEN_PACKET)
assert gen_manifest['specification']==spec, 'Use a new generation packet name after changing protocol'
print('Generation packet verified:',digest(GEN_PACKET/'manifest.json'))


## Select the API model and authorize one generation attempt
Add your API key to Colab Secrets as `OPENAI_API_KEY` and grant this notebook
access, or enter it through the hidden prompt. Set the exact API identifier for
the Astra model available to your account. No API model alias is assumed.


In [ ]:
MODEL_ID = ""  # Required: your exact Astra API model ID.
REASONING_EFFORT = None  # Example: 'high', only if this model supports it.
ATTEMPT_ID = "attempt-001"  # Predeclare attempts; do not select them by test scores.
MAX_INSPECTION_CALLS=40
MAX_TOOL_OUTPUT_CHARS=200_000
MAX_API_TURNS=45
MAX_OUTPUT_TOKENS_PER_TURN=8000

assert MODEL_ID, 'Set MODEL_ID before executing this cell'
assert '/' not in ATTEMPT_ID and '..' not in ATTEMPT_ID
from google.colab import userdata
import getpass
try: api_key=userdata.get('OPENAI_API_KEY')
except Exception: api_key=getpass.getpass('OpenAI API key (hidden): ')
assert api_key
os.environ['OPENAI_API_KEY']=api_key
del api_key
# Optional nonbillable model-list request. It checks visibility, not every capability.
from wsp_stopper.agent import API
ids={m['id'] for m in API().request('GET','models')['data']}
assert MODEL_ID in ids, 'Requested model is not visible to this API key; check the ID/account'
print('Selected model:',MODEL_ID)


In [ ]:
from wsp_stopper.agent import run
ATTEMPT=EXPERIMENT/'generation'/ATTEMPT_ID
ATTEMPT.parent.mkdir(exist_ok=True)
WEIGHTS=ATTEMPT/'weights.json'
if ATTEMPT.exists():
    assert WEIGHTS.exists(), 'Previous attempt did not finish. Preserve it; explicitly declare a new attempt to try again.'
    prior=json.loads((ATTEMPT/'run.json').read_text())
    assert prior['manifest_sha256']==digest(GEN_PACKET/'manifest.json')
    assert prior['model']==MODEL_ID
    print('Reusing the existing frozen controller; no API call.')
else:
    run(GEN_PACKET,PROJECT/'prompts/generate.md',MODEL_ID,ATTEMPT,
        max_calls=MAX_INSPECTION_CALLS,max_chars=MAX_TOOL_OUTPUT_CHARS,
        max_turns=MAX_API_TURNS,output_tokens=MAX_OUTPUT_TOKENS_PER_TURN,
        reasoning=REASONING_EFFORT)
from wsp_stopper.core import validate_weights
validate_weights(json.loads(WEIGHTS.read_text()))
WEIGHTS_SHA=digest(WEIGHTS)
print('Frozen controller SHA:',WEIGHTS_SHA)
# Keep a receipt for future re-evaluation; never modify weights after results.
receipt=ATTEMPT/'accepted_controller.json'
if receipt.exists(): assert json.loads(receipt.read_text())['weights_sha256']==WEIGHTS_SHA
else: write_json(receipt,dict(weights_sha256=WEIGHTS_SHA,tasks_sha256=digest(TASKS),
    packet_sha256=digest(GEN_PACKET/'manifest.json'),model=MODEL_ID))


## Evaluate after the weights are frozen
A fixed-cap baseline is repeated on the current GPU, followed by the WSP
controller. Both runs share the exact saved start/goal pairs, planner seeds,
model, objective, microbatching, and episode limits. This is serial execution;
logical iteration savings are reported alongside actual measured planning time.


In [ ]:
CONTROL=EXPERIMENT/f'baseline_recheck_{ATTEMPT_ID}.json'
ADAPTIVE=EXPERIMENT/f'wsp_{ATTEMPT_ID}.json'
for output,controller in [(CONTROL,None),(ADAPTIVE,WEIGHTS)]:
    if output.exists():
        saved=json.loads(output.read_text())
        assert saved['tasks_sha256']==digest(TASKS) and saved['runtime_sha256']==RUNTIME_SHA
        assert saved['weights_sha256']==(WEIGHTS_SHA if controller else None)
        print('Keeping completed evaluation:',output.name)
        continue
    args=[RUNTIME,'evaluate','--work',WORK,'--tasks',TASKS,'--out',output,
          '--iterations',30,'--population',POPULATION,'--elites',ELITES,'--microbatch',MICROBATCH]
    if controller: args += ['--weights',controller]
    run_python(*args)
assert digest(WEIGHTS)==WEIGHTS_SHA


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
fixed=json.loads(CONTROL.read_text()); adaptive=json.loads(ADAPTIVE.read_text())
assert fixed['config']==adaptive['config']
assert fixed['release_lock']==adaptive['release_lock']==protocol['checkpoint_lock']
assert fixed['device']==adaptive['device'] and fixed['torch_version']==adaptive['torch_version']
assert [r['id'] for r in fixed['episodes']]==[r['id'] for r in adaptive['episodes']]
a=pd.DataFrame(fixed['episodes']).set_index('id');b=pd.DataFrame(adaptive['episodes']).set_index('id')
paired=pd.DataFrame(dict(baseline_success=a.success,wsp_success=b.success,
    baseline_depth=a.depth,wsp_depth=b.depth,
    baseline_seconds=a.planning_seconds,wsp_seconds=b.planning_seconds,
    baseline_distance=a.final_distance,wsp_distance=b.final_distance))
summary=[]
for name,df in [('fixed_30',a),('wsp',b)]:
    summary.append(dict(method=name,episodes=len(df),success_rate=df.success.mean(),
        mean_depth=df.depth.mean(),mean_planning_seconds=df.planning_seconds.mean(),
        mean_transition_evals=(df.candidate_transition_evals+df.mean_transition_evals).mean(),
        mean_utility=df.utility.mean()))
summary=pd.DataFrame(summary).set_index('method')
display(summary);display(paired)
# Paired bootstrap over episodes, conditional on this one generated controller.
delta=b.success.astype(float).to_numpy()-a.success.astype(float).to_numpy()
rng=np.random.default_rng(7)
bootstrap=delta[rng.integers(0,len(delta),size=(2000,len(delta)))].mean(axis=1)
ci=np.quantile(bootstrap,[.025,.975]).tolist()
comparison=dict(success_delta=float(delta.mean()),paired_bootstrap_95=ci,
    discordant_wsp_wins=int(((~a.success)&b.success).sum()),
    discordant_wsp_losses=int((a.success&(~b.success)).sum()),
    iteration_saving=float(1-b.depth.sum()/a.depth.sum()),
    planning_time_saving=float(1-b.planning_seconds.sum()/a.planning_seconds.sum()),
    weights_sha256=WEIGHTS_SHA,tasks_sha256=digest(TASKS),
    caveat='Small pilot; CI covers episode sampling only, not generation variance')
# Check if saved notebook-01 baseline reproduced behavior on this session.
old=pd.DataFrame(baseline_saved['episodes']).set_index('id')
comparison['baseline_recheck_success_changes']=int((old.success!=a.success).sum())
summary.to_csv(EXPERIMENT/f'summary_{ATTEMPT_ID}.csv')
paired.to_csv(EXPERIMENT/f'paired_{ATTEMPT_ID}.csv')
write_json(EXPERIMENT/f'comparison_{ATTEMPT_ID}.json',comparison)
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,col,title in zip(axes,['success_rate','mean_depth','mean_planning_seconds'],['Success','CEM iterations','Planning seconds']):
    summary[col].plot.bar(ax=ax,rot=0,title=title)
axes[0].set_ylim(0,1)
fig.tight_layout();fig.savefig(EXPERIMENT/f'comparison_{ATTEMPT_ID}.png',dpi=160);plt.show()
print(json.dumps(comparison,indent=2))
print('All artifacts:',EXPERIMENT)


## Interpretation
Beating only the full depth-30 baseline does not establish superiority over the
best fixed budget. Add a predeclared fixed-depth sweep and held-out validation
before stronger claims. A controller always stopping at one depth is effectively
a fixed budget. Report failed generation attempts and every independently
generated controller; do not retain only the winner. This pilot compares fixed
budget and WSP, not the meeting-plan learned controller (training remains separate).

No test performance should be sent back to Astra to repair these weights under
the one-shot condition. A feedback-assisted follow-up is a separate experiment.
